In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from typing import List, Dict
import json
from datetime import datetime

In [2]:
# Load environment variables
load_dotenv()

# Get API key
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in environment variables. Please check your .env file.")

print("✓ API Key loaded successfully")

✓ API Key loaded successfully


In [10]:
# Initialize Groq LLM with gpt-oss120b model
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7,
    max_tokens=1024,
    groq_api_key=GROQ_API_KEY
)

print("✓ LLM initialized successfully")
print(f"Model: openai/gpt-oss120b")

✓ LLM initialized successfully
Model: openai/gpt-oss120b


In [4]:
class ConversationalAgent:
    def __init__(self, llm):
        self.llm = llm
        self.conversation_history = []  # Stores all messages
        self.conversation_summary = ""  # Rolling summary of conversation

        
    def _create_context_prompt(self, user_input: str) -> List:
        """Create the prompt with summary context + user input"""
        messages = []
        
        # Add system message
        system_prompt = "You are a helpful AI assistant. You have access to a summary of previous conversation context."
        messages.append(SystemMessage(content=system_prompt))
        
        # Add conversation summary if it exists
        if self.conversation_summary:
            summary_message = f"Previous conversation summary:\n{self.conversation_summary}"
            messages.append(SystemMessage(content=summary_message))
        
        # Add current user input
        messages.append(HumanMessage(content=user_input))
        
        return messages
    
    def _summarize_conversation(self) -> str:
        """Summarize the entire conversation history to extract essential information"""
        if not self.conversation_history:
            return ""
        
        # Format conversation for summarization
        conversation_text = ""
        for entry in self.conversation_history:
            conversation_text += f"User: {entry['user']}\n"
            conversation_text += f"Assistant: {entry['assistant']}\n\n"
        
        # Create summarization prompt
        summary_prompt = f"""Summarize the following conversation, keeping only the essential information, key topics discussed, important facts mentioned, and any decisions or conclusions made. Be concise but comprehensive.

Conversation:
{conversation_text}

Summary:"""
        
        # Get summary from LLM
        messages = [HumanMessage(content=summary_prompt)]
        response = self.llm.invoke(messages)
        
        return response.content
    
    def chat(self, user_input: str) -> str:
        """Main chat method - implements the full workflow"""
        # Step 1: Create context with summary + user input
        messages = self._create_context_prompt(user_input)
        
        # Step 2: Get response from LLM
        response = self.llm.invoke(messages)
        assistant_response = response.content
        
        # Step 3: Update conversation history
        self.conversation_history.append({
            "user": user_input,
            "assistant": assistant_response,
            "timestamp": datetime.now().isoformat()
        })
        
        # Step 4: Summarize conversation
        self.conversation_summary = self._summarize_conversation()
        
        return assistant_response
    
    def get_conversation_history(self) -> List[Dict]:
        """Return the full conversation history"""
        return self.conversation_history
    
    def get_summary(self) -> str:
        """Return the current conversation summary"""
        return self.conversation_summary
    
    def save_conversation(self, filename: str = "conversation_log.json"):
        """Save conversation history to a JSON file"""
        data = {
            "conversation_history": self.conversation_history,
            "summary": self.conversation_summary
        }
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        
        print(f"✓ Conversation saved to {filename}")
    
    def load_conversation(self, filename: str = "conversation_log.json"):
        """Load conversation history from a JSON file"""
        try:
            with open(filename, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            self.conversation_history = data.get("conversation_history", [])
            self.conversation_summary = data.get("summary", "")
            
            print(f"✓ Conversation loaded from {filename}")
        except FileNotFoundError:
            print(f"✗ File {filename} not found")
    
    def reset(self):
        """Reset the conversation"""
        self.conversation_history = []
        self.conversation_summary = ""
        print("✓ Conversation reset")

In [11]:
agent = ConversationalAgent(llm)
print("✓ Conversational Agent initialized")

✓ Conversational Agent initialized


In [12]:
# Turn 1
message1 = "Hi! My name is Bavly and I'm a AI engineer working on AI projects."
response1 = agent.chat(message1)
print(f"User: {message1}")
print(f"Assistant: {response1}")
print("\n" + "="*80 + "\n")

User: Hi! My name is Bavly and I'm a AI engineer working on AI projects.
Assistant: Hello Bavly! 👋 Nice to meet you. It’s great to connect with an AI engineer. What kind of AI projects are you working on right now? Anything exciting you’d like to share or discuss?




In [13]:
# Turn 2
message2 = "I'm currently building a chatbot with memory management. What are the key challenges?"
response2 = agent.chat(message2)
print(f"User: {message2}")
print(f"Assistant: {response2}")
print("\n" + "="*80 + "\n")

User: I'm currently building a chatbot with memory management. What are the key challenges?
Assistant: Building a chatbot that can *remember* things across turns (and even across sessions) is a powerful capability, but it also brings a whole set of design and engineering challenges. Below is a rundown of the most common hurdles you’ll encounter, grouped by theme, along with a few practical tips for each.

---

## 1. Memory Architecture & Scope  

| Challenge | Why it matters | Typical solutions / tips |
|-----------|----------------|--------------------------|
| **Short‑term vs. long‑term memory** | Short‑term (the last few turns) keeps the conversation coherent; long‑term (user preferences, facts) enables personalization. Mixing them can cause the model to “forget” the immediate context or to over‑emphasize stale information. | • Use a **hierarchical cache**: a sliding window of the last *N* turns for short‑term, and a separate structured store (e.g., key‑value DB) for long‑term facts

In [14]:
# Turn 3
message3 = "I'm 21 years old."
response3 = agent.chat(message3)
print(f"User: {message3}")
print(f"Assistant: {response3}")
print("\n" + "="*80 + "\n")

User: I'm 21 years old.
Assistant: Thanks for letting me know! 😊  
Is there anything specific you’d like to chat about or any project you’re working on that I can help with?




In [15]:
print("=== CURRENT CONVERSATION SUMMARY ===")
print(agent.get_summary())
print("\n" + "="*80)

=== CURRENT CONVERSATION SUMMARY ===
**Summary of Conversation**

- **Participants**
  - *User (Bavly)*: AI engineer, 21 years old, working on a chatbot with memory management.
  - *Assistant*: Provides guidance and acknowledges user information.

- **Key Topic Discussed**
  - **Challenges in building a chatbot with memory management**, organized into three main areas:

    1. **Memory Architecture & Scope**
       - Distinguish short‑term vs. long‑term memory; use hierarchical caches or separate stores.
       - Manage context‑window limits via summarization and Retrieval‑Augmented Generation (RAG).
       - Define a schema for “memorable” events; filter what to store.
       - Implement temporal decay, timestamps, and user‑initiated resets for forgetting.

    2. **Retrieval & Relevance**
       - Use vector embeddings and metadata filters to fetch relevant memories.
       - Balance precision/recall by retrieving a small top‑k set and re‑ranking.
       - Reduce latency with caching

In [16]:
print("=== FULL CONVERSATION HISTORY ===")
history = agent.get_conversation_history()
for i, turn in enumerate(history, 1):
    print(f"\n--- Turn {i} ({turn['timestamp']}) ---")
    print(f"User: {turn['user']}")
    print(f"Assistant: {turn['assistant']}")

=== FULL CONVERSATION HISTORY ===

--- Turn 1 (2026-09-14T13:54:19.260451) ---
User: Hi! My name is Bavly and I'm a AI engineer working on AI projects.
Assistant: Hello Bavly! 👋 Nice to meet you. It’s great to connect with an AI engineer. What kind of AI projects are you working on right now? Anything exciting you’d like to share or discuss?

--- Turn 2 (2026-09-14T13:55:22.693813) ---
User: I'm currently building a chatbot with memory management. What are the key challenges?
Assistant: Building a chatbot that can *remember* things across turns (and even across sessions) is a powerful capability, but it also brings a whole set of design and engineering challenges. Below is a rundown of the most common hurdles you’ll encounter, grouped by theme, along with a few practical tips for each.

---

## 1. Memory Architecture & Scope  

| Challenge | Why it matters | Typical solutions / tips |
|-----------|----------------|--------------------------|
| **Short‑term vs. long‑term memory** | Shor

In [19]:
def interactive_chat():
    print("=== Interactive Chat (type 'quit' to exit) ===")
    print("Type 'summary' to see current conversation summary")
    print("Type 'reset' to start a new conversation\n")
    
    while True:
        user_input = input("You: ").strip()
        
        if user_input.lower() == 'quit':
            print("Goodbye!")
            break
        
        if user_input.lower() == 'summary':
            print(f"\n--- Summary ---\n{agent.get_summary()}\n")
            continue
        
        if user_input.lower() == 'reset':
            agent.reset()
            continue
        
        if not user_input:
            continue
        
        # Get response
        response = agent.chat(user_input)
        print(f"\nAssistant: {response}\n")

# Uncomment to run interactive chat
interactive_chat()

=== Interactive Chat (type 'quit' to exit) ===
Type 'summary' to see current conversation summary
Type 'reset' to start a new conversation


Assistant: Your favorite football team is **Barcelona**.

Goodbye!


In [20]:
# Save the conversation to a file
agent.save_conversation("my_conversation.json")

✓ Conversation saved to my_conversation.json


In [ ]:
# Load a saved conversation
# agent.load_conversation("my_conversation.json")